# CustomerChurnX - Carga y validación inicial de datos

## Objetivo

Este notebook centraliza la lectura inicial del dataset oficial del proyecto y
valida que la fuente de datos esté disponible, sea legible y conserve una
estructura consistente para las etapas posteriores del pipeline MLOps.

La revisión se limita a controles de integridad básicos: dimensiones, vista
previa, tipos de datos, valores nulos, registros duplicados, validación del
esquema esperado y validación de rangos plausibles para variables críticas.
No se realiza análisis exploratorio profundo, limpieza, imputación ni
transformación de variables en esta etapa: esas decisiones se documentan y
ejecutan en `comprension_eda.ipynb` y en la etapa de Feature Engineering.

## Contrato de entrada / salida

- **Entrada:** `Base_de_datos.xlsx - Hoja1.csv`, ubicado un nivel por encima
  de este notebook dentro del repositorio del proyecto.
- **Salida:** un `DataFrame` `df` validado en esquema, junto con tablas de
  diagnóstico (nulos, duplicados, rangos) que documentan el estado inicial
  de calidad de los datos. Este notebook no persiste archivos intermedios;
  la siguiente etapa (`comprension_eda.ipynb`) vuelve a cargar el CSV
  original y reutiliza las mismas reglas de validación aquí definidas.

## Librerías

Se utilizan librerías estándar para administración de rutas, carga tabular
y trazabilidad de ejecución. La dependencia principal es `pandas`, porque
el dataset oficial se entrega como archivo CSV y debe cargarse con
`pd.read_csv`. Se añade `logging` en lugar de `print` para que los mensajes
de esta etapa sean compatibles con un orquestador de pipeline (Airflow,
Prefect, cron) y no solo con la ejecución interactiva en Jupyter.

In [1]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Iterable

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("customerchurnx.carga_datos")

## Configuración

La ruta se define de forma relativa a la ubicación del notebook. Esta
decisión mantiene el notebook portable dentro de la estructura oficial del
proyecto y evita dependencias con rutas absolutas del equipo local. A
diferencia de la versión anterior, **no se imprime la ruta absoluta
resuelta** en ningún mensaje de salida, para no filtrar información del
entorno local (usuario, sistema operativo, estructura de carpetas) en un
notebook que puede versionarse.

También se centralizan aquí el esquema esperado del dataset y las reglas de
rango válido para variables numéricas críticas. Estas reglas surgen de
hallazgos concretos: en la revisión inicial se detectaron valores
imposibles como edades de más de 120 años, puntajes negativos y un puntaje
DataCrédito por fuera del rango típico (150-950 en el contexto
colombiano). Documentarlas aquí, en la etapa de carga, permite detectarlas
tan pronto como sea posible en el pipeline.

In [2]:
DATA_PATH = Path("../Base_de_datos.xlsx - Hoja1.csv")
RANDOM_STATE = 42  # constante de reproducibilidad compartida con etapas futuras

# Esquema esperado del dataset.
# Se utilizará para validar que el archivo recibido
# contenga exactamente las variables requeridas, en el orden esperado.
EXPECTED_SCHEMA = [
    "tipo_credito",
    "fecha_prestamo",
    "capital_prestado",
    "plazo_meses",
    "edad_cliente",
    "tipo_laboral",
    "salario_cliente",
    "total_otros_prestamos",
    "cuota_pactada",
    "puntaje",
    "puntaje_datacredito",
    "cant_creditosvigentes",
    "huella_consulta",
    "saldo_mora",
    "saldo_total",
    "saldo_principal",
    "saldo_mora_codeudor",
    "creditos_sectorFinanciero",
    "creditos_sectorCooperativo",
    "creditos_sectorReal",
    "promedio_ingresos_datacredito",
    "tendencia_ingresos",
    "Pago_atiempo",
]

# Reglas de rango plausible para variables numéricas críticas.
# Fuente: hallazgos de la revisión técnica inicial del dataset.
VALUE_RANGE_RULES = {
    "edad_cliente": (18, 100),
    "puntaje": (0, 100),
    "puntaje_datacredito": (150, 950),
}

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

## Funciones del pipeline de carga

Se encapsula la lógica de carga y validación en funciones puras, con
docstrings y *type hints*, para que puedan reutilizarse desde
`comprension_eda.ipynb` o desde un script de producción sin copiar y pegar
código entre notebooks (principio DRY, requisito básico de un pipeline
MLOps mantenible).

A diferencia de la versión anterior, la validación de esquema ahora
**detiene la ejecución** (`raise ValueError`) si el archivo recibido no
coincide con el esquema esperado, con el mismo criterio de "fail fast" que
ya se aplicaba correctamente a la ausencia del archivo.

In [3]:
def validate_file_exists(path: Path) -> None:
    """Verifica que el archivo fuente exista antes de intentar leerlo.

    Lanza FileNotFoundError si el archivo no está disponible en la ruta
    relativa esperada. No expone la ruta absoluta del sistema local en el
    mensaje de éxito, para mantener el notebook libre de información de
    entorno cuando se versiona.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró el dataset oficial en la ruta relativa: {path}\n"
            f"Directorio de trabajo actual: {Path.cwd()}\n"
            f"Ruta absoluta esperada: {path.resolve()}\n"
            "Verifique que el notebook se ejecute desde la carpeta "
            "'notebooks/' del repositorio y que el archivo "
            "'Base_de_datos.xlsx - Hoja1.csv' exista un nivel arriba de esa "
            "carpeta. Si su estructura de carpetas es distinta, actualice "
            "DATA_PATH en la celda de configuración."
        )
    logger.info("Archivo fuente localizado correctamente en: %s", path)


def load_dataset(path: Path) -> pd.DataFrame:
    """Carga el dataset oficial desde un archivo CSV.

    Aplica validate_file_exists antes de leer, y registra las dimensiones
    resultantes en el log del pipeline.
    """
    validate_file_exists(path)
    df = pd.read_csv(path)
    logger.info(
        "Dataset cargado: %s registros, %s variables.", df.shape[0], df.shape[1]
    )
    return df


def validate_schema(df: pd.DataFrame, expected_schema: Iterable[str]) -> pd.DataFrame:
    """Valida que las columnas del DataFrame coincidan con el esquema oficial.

    Lanza ValueError si existen columnas faltantes o no esperadas. Si las
    columnas coinciden mas no respetan el orden exacto, se registra una
    advertencia (no se considera un error bloqueante, porque el orden de
    columnas de un CSV no siempre está garantizado por la fuente).
    """
    expected_schema = list(expected_schema)
    observed_columns = list(df.columns)
    missing_columns = sorted(set(expected_schema) - set(observed_columns))
    unexpected_columns = sorted(set(observed_columns) - set(expected_schema))
    schema_is_valid = observed_columns == expected_schema

    schema_validation = pd.DataFrame(
        {
            "validacion": [
                "columnas_en_orden_esperado",
                "columnas_faltantes",
                "columnas_no_esperadas",
            ],
            "resultado": [schema_is_valid, missing_columns, unexpected_columns],
        }
    )

    if missing_columns or unexpected_columns:
        raise ValueError(
            f"Esquema inválido. Columnas faltantes: {missing_columns}. "
            f"Columnas no esperadas: {unexpected_columns}."
        )
    if not schema_is_valid:
        logger.warning(
            "Las columnas coinciden pero no respetan el orden esperado del esquema."
        )

    logger.info("Esquema validado correctamente.")
    return schema_validation


def validate_value_ranges(df: pd.DataFrame, rules: dict) -> pd.DataFrame:
    """Cuantifica, por variable, cuántos registros violan un rango plausible.

    No elimina ni corrige valores: esta etapa solo diagnostica y documenta
    el hallazgo para que sea tratado explícitamente en Feature Engineering.
    """
    rows = []
    for column, (lower, upper) in rules.items():
        if column not in df.columns:
            continue
        series = df[column]
        violations = series[(series < lower) | (series > upper)]
        rows.append(
            {
                "variable": column,
                "rango_valido": f"[{lower}, {upper}]",
                "valores_fuera_de_rango": int(violations.shape[0]),
                "porcentaje": round(violations.shape[0] / len(df) * 100, 2),
            }
        )
    return pd.DataFrame(rows).sort_values("valores_fuera_de_rango", ascending=False)

## Ejecución: carga y validación de esquema

Se ejecuta el pipeline de carga y, de forma inmediata, la validación de
esquema. Si el esquema no es válido, la ejecución se detiene aquí mismo:
ninguna celda posterior debería ejecutarse sobre una fuente de datos
estructuralmente incorrecta.

In [4]:
df = load_dataset(DATA_PATH)
schema_validation = validate_schema(df, EXPECTED_SCHEMA)
schema_validation

INFO | Archivo fuente localizado correctamente en: ..\Base_de_datos.xlsx - Hoja1.csv
INFO | Dataset cargado: 10763 registros, 23 variables.
INFO | Esquema validado correctamente.


,validacion,resultado
0,columnas_en_orden_esperado,True
1,columnas_faltantes,[]
2,columnas_no_esperadas,[]


## Dimensiones del dataset

La validación de dimensiones confirma el volumen inicial de observaciones
y variables que alimentará los siguientes notebooks del proyecto.

In [5]:
dataset_shape = {"registros": df.shape[0], "variables": df.shape[1]}
print(dataset_shape)

{'registros': 10763, 'variables': 23}


## Información general

La información general permite revisar de forma consolidada la cantidad
de registros no nulos por variable, el tipo inferido por `pandas` y el uso
de memoria del DataFrame.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tipo_credito                   10763 non-null  int64  
 1   fecha_prestamo                 10763 non-null  str    
 2   capital_prestado               10763 non-null  int64  
 3   plazo_meses                    10763 non-null  int64  
 4   edad_cliente                   10763 non-null  int64  
 5   tipo_laboral                   10763 non-null  str    
 6   salario_cliente                10763 non-null  int64  
 7   total_otros_prestamos          10763 non-null  int64  
 8   cuota_pactada                  10763 non-null  int64  
 9   puntaje                        10763 non-null  float64
 10  puntaje_datacredito            10757 non-null  float64
 11  cant_creditosvigentes          10763 non-null  int64  
 12  huella_consulta                10763 non-null  int64  
 1

## Tipos de datos

La inspección de tipos permite identificar variables numéricas, variables
textuales y posibles campos que requieran conversión en etapas
posteriores. En este notebook solo se reportan los tipos inferidos; no se
transforman columnas.

In [7]:
data_types = (
    df.dtypes.astype(str)
    .rename("tipo_dato")
    .reset_index()
    .rename(columns={"index": "variable"})
)
data_types

,variable,tipo_dato
0,tipo_credito,int64
1,fecha_prestamo,str
2,capital_prestado,int64
3,plazo_meses,int64
4,edad_cliente,int64
5,tipo_laboral,str
6,salario_cliente,int64
7,total_otros_prestamos,int64
8,cuota_pactada,int64
9,puntaje,float64


## Valores nulos

El conteo de valores nulos permite dimensionar la completitud del dataset
antes del análisis exploratorio. La definición de tratamiento de nulos se
reserva para etapas posteriores; aquí solo se documenta la situación
inicial.

In [8]:
missing_values = (
    df.isna()
    .sum()
    .rename("nulos")
    .reset_index()
    .rename(columns={"index": "variable"})
)
missing_values["porcentaje"] = (missing_values["nulos"] / len(df) * 100).round(2)
missing_values = (
    missing_values.query("nulos > 0")
    .sort_values("nulos", ascending=False)
    .reset_index(drop=True)
)
missing_values

,variable,nulos,porcentaje
0,tendencia_ingresos,2932,27.24
1,promedio_ingresos_datacredito,2930,27.22
2,saldo_mora_codeudor,590,5.48
3,saldo_principal,405,3.76
4,saldo_mora,156,1.45
5,saldo_total,156,1.45
6,puntaje_datacredito,6,0.06


## Registros duplicados

La revisión de duplicados identifica si existen filas completamente
repetidas. Este control es una validación de calidad; no implica
eliminación de registros en este notebook.

In [9]:
duplicate_rows = int(df.duplicated().sum())
print({"registros_duplicados": duplicate_rows})

{'registros_duplicados': 0}


## Validación de rangos válidos

Además de la validación estructural (esquema), se valida que un
subconjunto de variables numéricas críticas se encuentre dentro de un
rango plausible según el negocio: `edad_cliente` no debería superar los
100 años, `puntaje` está definido en una escala de 0 a 100, y
`puntaje_datacredito` corresponde a un score de buró cuyo rango típico en
Colombia es 150-950. Esta validación **no elimina ni corrige** registros;
únicamente cuantifica el hallazgo para que se trate explícitamente en la
etapa de comprensión exploratoria y en Feature Engineering. No tratar esto
en silencio evita que valores imposibles se interpreten más adelante como
señales estadísticas válidas.

In [10]:
range_validation = validate_value_ranges(df, VALUE_RANGE_RULES)
range_validation

,variable,rango_valido,valores_fuera_de_rango,porcentaje
2,puntaje_datacredito,"[150, 950]",153,1.42
0,edad_cliente,"[18, 100]",150,1.39
1,puntaje,"[0, 100]",135,1.25


## Conclusiones

La validación inicial confirma que el dataset oficial está disponible y se
carga correctamente con `pandas`, utilizando una ruta relativa que
preserva la portabilidad del proyecto dentro de la estructura del
repositorio, sin exponer rutas absolutas del entorno local en los mensajes
de salida.

El conjunto de datos está conformado por **10.763 registros** y **23
variables**, sin evidencia de registros completamente duplicados. La
validación de esquema, ahora bloqueante, confirma que todas las columnas
esperadas están presentes; si en una ejecución futura el esquema cambiara,
el pipeline se detendrá de forma explícita en esta celda en lugar de
propagar una fuente de datos inconsistente a las siguientes etapas.

Se identificaron valores faltantes concentrados principalmente en
`tendencia_ingresos` y `promedio_ingresos_datacredito` (ambas alrededor de
27%), y en menor medida en variables de saldo y mora. La coincidencia de
magnitud entre las dos primeras variables sugiere un origen común que debe
investigarse en la etapa de comprensión exploratoria (por ejemplo, un
segmento de clientes sin historial de buró).

La validación de rangos detectó valores fuera de los límites plausibles en
`edad_cliente`, `puntaje` y `puntaje_datacredito`. Estos casos no se
corrigen en esta etapa: quedan documentados como hallazgos de calidad de
datos que deben tratarse explícitamente antes del modelado, ya sea como
errores de captura a corregir o como reglas de imputación específicas.

Finalmente, se verifica que la variable objetivo disponible corresponde a
**`Pago_atiempo`**, por lo que el proyecto utilizará esta columna como
referencia para las fases posteriores de análisis, ingeniería de
características y modelado predictivo.